# Gold runner
1. Run the first cell, then fill in the two boxes at the top.
2. Run the remaining cells (or Run all).

First ever Gold run: use `load_type = full`. (An incremental run on an empty Gold fact table switches to full by itself.)

In [0]:
# CELL 1 - run this first. It only creates the two input boxes at the top of the notebook.
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("processing_date", "")

In [0]:
# Gold pipeline runner with failure logging
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
from delta.tables import DeltaTable

# (the two input boxes are created in the first cell)
load_type = dbutils.widgets.get("load_type")
processing_date = dbutils.widgets.get("processing_date").strip()

runner_run_id = str(uuid.uuid4())
runner_start_time = datetime.now(timezone.utc).replace(tzinfo=None)
log_table = "workspace.default.pipeline_execution_logs"

log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("processing_date", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("error_message", StringType(), True),
])

try:
    print("Starting Gold pipeline...")
    result = dbutils.notebook.run(
        "./03_gold_pipeline_final",
        0,
        {"load_type": load_type, "processing_date": processing_date},
    )
    print("Gold pipeline completed successfully!")
    print("Notebook result:", result)

except Exception as e:
    failure_df = spark.createDataFrame(
        [(
            runner_run_id, "gold", "workspace.default.gold_fact_market_daily",
            load_type, processing_date or None, "workspace.default.silver_market_observation",
            runner_start_time, datetime.now(timezone.utc).replace(tzinfo=None),
            "FAILED", 0, 0, 0, 0, str(e)[:4000],
        )],
        schema=log_schema,
    )
    (
        DeltaTable.forName(spark, log_table).alias("target")
        .merge(failure_df.alias("source"), "target.run_id = source.run_id")
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Gold pipeline FAILED. Failure recorded in execution logs.")
    raise

In [0]:
# Latest Gold execution logs (one row per Gold table)
display(
    spark.table("workspace.default.pipeline_execution_logs")
    .filter(F.col("layer") == "gold")
    .orderBy(F.col("end_timestamp").desc())
    .select("table_name", "load_type", "processing_date", "status", "rows_read",
            "rows_inserted", "rows_updated", "start_timestamp", "end_timestamp")
    .limit(8)
)

In [0]:
# Consistency checks on the star schema (primary keys, foreign keys, completeness)
fact = spark.table("workspace.default.gold_fact_market_daily")
dim_coin = spark.table("workspace.default.gold_dim_coin")
dim_date = spark.table("workspace.default.gold_dim_date")
silver = spark.table("workspace.default.silver_market_observation")

fact_count = fact.count()
fact_duplicate_keys = fact.groupBy("coin_id", "observation_date").count().filter(F.col("count") > 1).count()
orphan_coins = fact.join(dim_coin, "coin_id", "left_anti").count()
orphan_dates = fact.join(dim_date, fact["observation_date"] == dim_date["full_date"], "left_anti").count()
silver_coin_days = silver.select("coin_id", F.to_date("observation_timestamp").alias("d")).distinct().count()

print("Fact rows:", fact_count, "| Silver coin-days:", silver_coin_days)
print("Duplicate fact keys:", fact_duplicate_keys)
print("Fact rows without a coin in dim_coin:", orphan_coins)
print("Fact rows without a date in dim_date:", orphan_dates)

if fact_duplicate_keys == 0 and orphan_coins == 0 and orphan_dates == 0 and fact_count == silver_coin_days:
    print("PASS: Gold star schema is consistent with Silver.")
else:
    print("CHECK REQUIRED: if the fact count is lower than Silver coin-days, run Gold once with load_type = full.")